In [1]:
from spark_start import spark
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(PROJECT_ROOT)

df = spark.read.parquet(f"{PROJECT_ROOT}/data/parquet/network_data")
print(df.count())


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/16 00:40:06 WARN Utils: Your hostname, Saideep-Reddys-Mac.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.2 instead (on interface en0)
26/02/16 00:40:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/16 00:40:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/16 00:40:08 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/02/16 00:40:08 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Spark started successfully
2540047


In [2]:
df.printSchema()


root
 |-- proto: string (nullable = true)
 |-- state: string (nullable = true)
 |-- dur: double (nullable = true)
 |-- sbytes: integer (nullable = true)
 |-- dbytes: integer (nullable = true)
 |-- sttl: integer (nullable = true)
 |-- dttl: integer (nullable = true)
 |-- sloss: integer (nullable = true)
 |-- dloss: integer (nullable = true)
 |-- service: string (nullable = true)
 |-- sload: double (nullable = true)
 |-- dload: double (nullable = true)
 |-- spkts: integer (nullable = true)
 |-- dpkts: integer (nullable = true)
 |-- swin: integer (nullable = true)
 |-- dwin: integer (nullable = true)
 |-- stcpb: long (nullable = true)
 |-- dtcpb: long (nullable = true)
 |-- smeansz: integer (nullable = true)
 |-- dmeansz: integer (nullable = true)
 |-- trans_depth: integer (nullable = true)
 |-- res_bdy_len: integer (nullable = true)
 |-- sjit: double (nullable = true)
 |-- djit: double (nullable = true)
 |-- stime: integer (nullable = true)
 |-- ltime: integer (nullable = true)
 |-- sint

In [3]:
from pyspark.ml.feature import StringIndexer

categorical_cols = ["proto","state","service"]

indexers = [
    StringIndexer(inputCol=col, outputCol=col+"_idx", handleInvalid="keep")
    for col in categorical_cols
]


In [4]:
exclude = ["label","label_binary","attack_cat","proto","state","service"]
feature_cols = [c for c in df.columns if c not in exclude]

feature_cols[:10], len(feature_cols)


(['dur',
  'sbytes',
  'dbytes',
  'sttl',
  'dttl',
  'sloss',
  'dloss',
  'sload',
  'dload',
  'spkts'],
 40)

In [5]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_cols + [c+"_idx" for c in categorical_cols],
    outputCol="features_raw"
)


In [6]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features"
)


In [7]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=indexers + [assembler, scaler])

model = pipeline.fit(df)
processed = model.transform(df)


IllegalArgumentException: Data type string of column ct_ftp_cmd is not supported.

In [8]:
from pyspark.sql.types import StringType

string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
string_cols


['proto', 'state', 'service', 'ct_ftp_cmd', 'attack_cat']

In [9]:
categorical_cols = [c for c in string_cols if c not in ["attack_cat"]]
categorical_cols


['proto', 'state', 'service', 'ct_ftp_cmd']

In [10]:
from pyspark.ml.feature import StringIndexer

indexers = [
    StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep")
    for c in categorical_cols
]


In [11]:
exclude = ["label","label_binary","attack_cat"] + categorical_cols
feature_cols = [c for c in df.columns if c not in exclude]


In [12]:
pipeline = Pipeline(stages=indexers + [assembler, scaler])
model = pipeline.fit(df)
processed = model.transform(df)


IllegalArgumentException: Data type string of column ct_ftp_cmd is not supported.

In [13]:
pipeline = Pipeline(stages=indexers + [assembler, scaler])
model = pipeline.fit(df)
processed = model.transform(df)

# DROP original string columns
processed = processed.drop(*categorical_cols)


IllegalArgumentException: Data type string of column ct_ftp_cmd is not supported.

In [14]:
from pyspark.sql.types import StringType

string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
print("String columns:", string_cols)


String columns: ['proto', 'state', 'service', 'ct_ftp_cmd', 'attack_cat']


In [15]:
categorical_cols = [c for c in string_cols if c not in ["attack_cat"]]
print("Categorical:", categorical_cols)


Categorical: ['proto', 'state', 'service', 'ct_ftp_cmd']


In [16]:
exclude_cols = ["label", "label_binary", "attack_cat"] + categorical_cols

numeric_cols = [c for c in df.columns if c not in exclude_cols]
print("Numeric count:", len(numeric_cols))


Numeric count: 39


In [17]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=numeric_cols + [c + "_idx" for c in categorical_cols],
    outputCol="features_raw"
)


In [18]:
pipeline = Pipeline(stages=indexers + [assembler, scaler])
model = pipeline.fit(df)
processed = model.transform(df)

final_df = processed.select("features", "label_binary")


26/02/16 00:43:38 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

In [19]:
final_df.show(5)


+--------------------+------------+
|            features|label_binary|
+--------------------+------------+
|(43,[0,1,2,3,4,7,...|           0|
|(43,[0,1,2,3,4,7,...|           0|
|(43,[0,1,2,3,4,7,...|           0|
|(43,[0,1,2,3,4,7,...|           0|
|(43,[0,1,2,3,4,7,...|           0|
+--------------------+------------+
only showing top 5 rows


In [20]:
final_df.write.mode("overwrite").parquet(f"{PROJECT_ROOT}/data/processed/final")
